# 04 — Legacy-holdout comparability check

This notebook reports the outer-CV summary, then fits specifications locked before consulting the separately reported 80/20 scores and evaluates that previously used population for secondary comparability. Because primary outer CV uses every labelled row, legacy-holdout rows also participate in development folds; this surface is not presented as pristine or independent evidence.

In [1]:
from pathlib import Path
import json
import joblib
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.modeling import fit_legacy_holdout_models
from src.validation import (metric_row, risk_decile_table, capacity_table, plot_roc_pr,
    plot_calibration, plot_bad_rate_by_decile, plot_cumulative_capture, plot_capacity_tradeoff)

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"
MODELS = ROOT / "models"

features = pd.read_pickle(PROCESSED / "borrower_features.pkl")
development = joblib.load(MODELS / "development_result.joblib")
lock = json.loads((MODELS / "design_lock.json").read_text(encoding="utf-8"))
locked = fit_legacy_holdout_models(features, development)
legacy = locked["legacy_holdout"]
y_legacy = legacy["TARGET"].to_numpy()

In [2]:
metric_rows = []
for model, score in locked["predictions"].items():
    metric_rows.append(metric_row(y_legacy, score, model, "overall", False))

# Required subgroup checks for age and family status; no legal-fairness claim is made.
age_group = pd.cut(legacy["APP_AGE_YEARS"], [0, 30, 45, 60, np.inf], right=False, labels=["age_<30", "age_30_44", "age_45_59", "age_60+"])
for model, score in locked["predictions"].items():
    for label in age_group.dropna().unique():
        mask = age_group.eq(label).to_numpy()
        if y_legacy[mask].sum() >= 25 and (~y_legacy[mask].astype(bool)).sum() >= 25:
            metric_rows.append(metric_row(y_legacy[mask], score[mask], model, str(label), False))
    for label in legacy["APP_FAMILY_STATUS"].fillna("Missing").unique():
        mask = legacy["APP_FAMILY_STATUS"].fillna("Missing").eq(label).to_numpy()
        if y_legacy[mask].sum() >= 25 and (~y_legacy[mask].astype(bool)).sum() >= 25:
            metric_rows.append(metric_row(y_legacy[mask], score[mask], model, f"family_{label}", False))

legacy_metrics = pd.DataFrame(metric_rows)
deciles = pd.concat([risk_decile_table(y_legacy, score, model) for model, score in locked["predictions"].items()], ignore_index=True)
capacity = pd.concat([capacity_table(y_legacy, score, model) for model, score in locked["predictions"].items()], ignore_index=True)
top_decile_lift = deciles.loc[deciles["risk_decile"].eq(1)].set_index("model")["lift_vs_portfolio"]
capture_at_10pct = capacity.loc[capacity["capacity_share"].eq(0.10)].set_index("model")["bad_capture_share"]
overall = legacy_metrics["sample"].eq("overall")
legacy_metrics.loc[overall, "top_decile_lift"] = legacy_metrics.loc[overall, "model"].map(top_decile_lift)
legacy_metrics.loc[overall, "capture_at_10pct"] = legacy_metrics.loc[overall, "model"].map(capture_at_10pct)

legacy_metrics.to_csv(TABLES / "legacy_holdout_model_metrics.csv", index=False)
deciles.to_csv(TABLES / "risk_decile_summary.csv", index=False)
capacity.to_csv(TABLES / "capacity_analysis.csv", index=False)
locked["lightgbm_importance"].to_csv(TABLES / "challenger_feature_importance.csv", index=False)

development.logistic_stability.to_csv(TABLES / "logistic_coefficients.csv", index=False)

plot_roc_pr(y_legacy, locked["predictions"], FIGURES / "roc_pr_model_comparison.png")
plot_calibration(y_legacy, locked["predictions"], FIGURES / "calibration_comparison.png")
plot_bad_rate_by_decile(deciles, FIGURES / "bad_rate_by_decile.png")
plot_cumulative_capture(deciles, FIGURES / "cumulative_bad_capture.png")
plot_capacity_tradeoff(capacity, FIGURES / "capacity_tradeoff.png")

In [3]:
assert len(legacy) == len(development.split_ids["legacy_holdout"])
assert deciles.groupby("model")["borrowers"].sum().eq(len(legacy)).all()
assert deciles.groupby("model")["bads"].sum().eq(y_legacy.sum()).all()
assert deciles.groupby("model")["cumulative_bad_capture"].last().pipe(np.allclose, 1.0)
expected = [int(np.ceil(len(legacy) * x)) for x in [0.05, 0.10, 0.20]]
assert all(group["reviewed_borrowers"].tolist() == expected for _, group in capacity.groupby("model", sort=False))
display(development.outer_cv_summary)
display(legacy_metrics[legacy_metrics["sample"].eq("overall")])
display(capacity)
print(f"Outer-CV-selected primary model: {lock['primary_model']}")

,model,roc_auc_mean,roc_auc_std,gini_mean,gini_std,average_precision_mean,average_precision_std,ks_mean,ks_std,brier_mean,brier_std,capture_at_5pct_mean,capture_at_5pct_std,capture_at_10pct_mean,capture_at_10pct_std,capture_at_20pct_mean,capture_at_20pct_std
0,Application-only Logistic,0.657943,0.003843,0.315886,0.007686,0.145461,0.002803,0.235433,0.008657,0.072269,0.000104,0.125760,0.005142,0.221752,0.004277,0.378248,0.005466
1,Logistic,0.723163,0.001218,0.446326,0.002437,0.200443,0.003689,0.331930,0.002331,0.069852,0.000129,0.173978,0.005118,0.292085,0.004556,0.467472,0.002301
2,LightGBM,0.753450,0.002978,0.506899,0.005957,0.234641,0.002986,0.378581,0.006101,0.068238,0.000135,0.200282,0.002403,0.327009,0.001470,0.512910,0.004183


,model,sample,calibrated,roc_auc,gini,average_precision,ks,brier,calibration_slope,calibration_intercept,event_rate,rows,top_decile_lift,capture_at_10pct
0,Logistic,overall,False,0.724768,0.449536,0.202944,0.339162,0.069738,1.002760,0.007111,0.080728,61503,2.970457,0.297080
1,LightGBM,overall,False,0.755179,0.510357,0.237120,0.379754,0.068091,1.019758,0.048356,0.080728,61503,3.389342,0.338973


,model,capacity_share,reviewed_borrowers,population_share,captured_bads,observed_event_count,bad_capture_share,event_capture,cumulative_event_capture,reviewed_bad_rate,observed_event_rate,lift_vs_portfolio,non_events_reviewed,reviews_per_captured_event
0,Logistic,0.05,3076,0.050014,873,873,0.175831,0.175831,0.175831,0.283810,0.283810,3.515645,2203,3.523482
1,Logistic,0.10,6151,0.100011,1475,1475,0.297080,0.297080,0.297080,0.239798,0.239798,2.970457,4676,4.170169
2,Logistic,0.20,12301,0.200007,2358,2358,0.474924,0.474924,0.474924,0.191692,0.191692,2.374545,9943,5.216709
3,LightGBM,0.05,3076,0.050014,1006,1006,0.202618,0.202618,0.202618,0.327048,0.327048,4.051247,2070,3.057654
4,LightGBM,0.10,6151,0.100011,1683,1683,0.338973,0.338973,0.338973,0.273614,0.273614,3.389342,4468,3.654783
5,LightGBM,0.20,12301,0.200007,2602,2602,0.524068,0.524068,0.524068,0.211528,0.211528,2.620257,9699,4.727517


Outer-CV-selected primary model: LightGBM


The legacy holdout is the same reproducible random split evaluated by the earlier project version. It is retained only for continuity, not treated as independent or out-of-time evidence. Capacity rows describe risk concentration at prespecified review volumes; they are not optimized cutoffs and contain no assumed approval economics. Penalized Logistic coefficients are associative, not causal.